# Anwenderwebsite lokal bauen

Dieses How-to baut dieselben Lehrgrafiken, Downloads und Notebook-Lesefassungen wie Quarto. Die realen Beispiele werden aus `website/_examples.yml` ausgewählt. Voraussetzung: `python -m pip install -e ".[publication]"` im Repository, Quarto für die HTML-Seiten. Für die vorhandenen synthetischen Grafikexporte zusätzlich Chrome/Kaleido.

Die Website läuft statisch, beispielsweise auf GitHub Pages. Sie nutzt das bestehende Repository und Python-Paket beim Build; der Browser führt keine Python- oder Excel-Simulation aus. Eine Veröffentlichung ist ein separater Schritt.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/website_assets.py").exists()
sys.path.insert(0, str(ROOT / "scripts"))
from website_assets import build
output = build(ROOT / "notebooks/local_exports/website_tutorial")
assert (output / "assets/teaching/index.html").exists()
assert (output / "assets/components/index.html").exists()
print("Lehrmodule, öffentliche Unterlagen und Notebook-Lesefassungen erstellt.")

## Lehrgrafiken ausprobieren

Im ersten Modul: Bauteil auswählen, Hülle aktivieren, gekühlte/ungekühlte Zone vergleichen, einzelne Wärmeflüsse ausblenden, aktuellen Zustand als SVG speichern. Der zweite Schnitt zeigt eine ausdrücklich angenommene unbeheizte Kellerzone. Die Kategorienamen stammen aus dem bestehenden Schema; nicht vorhandene Elemente erhalten keine Ersatzzuordnung.

In [ ]:
from IPython.display import HTML, display
from component_teaching import categories
assert len(categories()) == 20
display(HTML('<iframe title="Bauteilkatalog" style="width:100%;height:850px" srcdoc="' + __import__('html').escape((output / "assets/components/index.html").read_text(encoding="utf-8"), quote=True) + '"></iframe>'))

## Quarto rendern und prüfen

Im Repository: `quarto render website`. Das deutsche Projekt baut die Hauptseiten; sein Post-Render-Schritt startet das englische Unterprojekt `website/en`. Beide Sprachfassungen landen gemeinsam in `website/_site`, die englischen Seiten unter `_site/en/`.

Die beiden `_quarto.yml`-Dateien definieren getrennte Kopf- und Seitenleisten. Die Quelldateien liegen im selben Sprachverzeichnis wie ihre Navigation. Beide Versionen lokal über die Website-Vorschau kontrollieren; nach Inhaltsänderungen den Build erneut ausführen.

Die bisherigen Notebook-Lesefassungen und alle Downloads enthalten keine gespeicherten Zellenausgaben. Das Bewertungstutorial übernimmt seine gespeicherten Ausgaben in eine native Quarto-Seite; die Originale bleiben unverändert. Quellen: `docs/`, `notebooks/`, `data/schemas/1.13.1_dev/IN.csv`. Generierte Veröffentlichungsdateien liegen in `website/_site`.

## Konfigurierte Ergebnisse erzeugen

Website und Publikations-Notebook verwenden denselben Adapter. Hier werden die interaktiven HTML-Ergebnisse ohne erneute statische Bildausgabe erzeugt.

In [ ]:
from website_examples import build as build_examples
reports, checks = build_examples(output / "assets/results", formats=("html",))
assert len(reports) == 4
checks

## Excel-Dokumentation und bereitgestellte Abbildungen

Die deutsche Navigation trennt Tool-Anwendung, Modellieren und Analysieren und Bewerten. Die englische Quarto-Unterseite definiert dieselbe Struktur mit eigener englischer Kopf- und Seitenleiste. `results.qmd` beschreibt Erg; `excel-dashboard.qmd` beschreibt das Excel-Dashboard; `dashboard.qmd` bleibt der Python-Szenarioexplorer. Migration, PV, Varianten und Scen besitzen eigene Seiten.

Originalbilder liegen unter `docs/website/`. Die explizite Liste `MANUAL_FIGURES` in `website_assets.py` ist die Veröffentlichungsauswahl. `build` kopiert sie unverändert nach `media/manual/`; keine veralteten Handbuch-Screenshots werden erzeugt. Bei Austausch Quellen-/Versionshinweis in der QMD-Bildunterschrift mitpflegen.

Bedienorte mit `.excel-menu` markieren, Pfade in `.excel-path`, Schritte in `.excel-operation`, Felder mit `.excel-field`. Fachliche Erläuterungen bleiben normaler Fließtext; Nachweisbindungen stehen in `.declaration-note`. Jede Abbildung benötigt Alt-Text und eine Quellenangabe. Die Marken sind keine Excel-Schaltflächen.

Beispiel: `**Excel-Ort:** Blatt [🔥 Heizung]{.excel-menu .darkred}`. Die vollständige Formatierungslegende steht in `excel-navigation.qmd`.


In [ ]:
from website_assets import MANUAL_FIGURES
for name in MANUAL_FIGURES:
    source = ROOT / "docs/website" / f"{name}.png"
    published = output / "media/manual" / source.name
    assert published.read_bytes() == source.read_bytes()
print(f"{len(MANUAL_FIGURES)} Originalabbildungen unverändert veröffentlicht.")


### Visuelle Abnahme

Nach `quarto render website` die Seiten Presets, Blattnavigator und Excel-Dashboard auf Desktop und bei 390 px Breite prüfen. Blattmarken und Pfade müssen umbrechen; Bilder sind auf weißem Hintergrund und per Lightbox vergrößerbar. Fehlende Ansichten sind als „In Vorbereitung“ gekennzeichnet. Linkziele und Anker einschließlich F7 unter `examples.qmd#aspern-f7` prüfen.


## English pages

English mirrors live in `website/en/` as a separate Quarto website project with `lang: en`. It defines its own English navbar and sidebar, while the German project keeps its German navigation. Keep German worksheet and ribbon labels where they identify controls in the workbook. English links point to pages within `en/`; images, downloads and generated visualisations use paths relative to the parent website. Running `quarto render website` renders the German site and then the English subproject into the shared `_site/en/` output. The English render is launched with `website/en` as its working directory so Quarto reads the English project configuration.


## Quellenstand und reproduzierbarer Build

`master` enthält die bearbeitbaren QMD-Seiten, beide Quarto-Konfigurationen, CSS, Python-Skripte, Notebooks und die benötigten Eingabedateien. `gh-pages` enthält ausschließlich die Veröffentlichung einschließlich der vorgesehenen Notebook-Downloads. Generierte Verzeichnisse unter `website/` bleiben ignoriert.

Vor Änderungen lokale, unversionierte und benötigte ignorierte Dateien separat sichern; ein Git-Bundle enthält nur Git-Objekte und Referenzen. Änderungen zunächst auf einem Entwicklungsbranch prüfen. Commit, Übernahme in `master`, Push und Veröffentlichung erfolgen nur nach der jeweiligen Freigabe.

Für einen frischen Build müssen insbesondere die drei `data/exports/Bsp_San_6G*.xlsx`, die Schulungs-PDF und -PPTX, `docs/ComponentTypes.png`, `docs/mobilität.png`, `docs/website/*.png` und `website/illustrations/peq-stufen.png` vorhanden sein. Die eng begrenzten Ignore-Ausnahmen erhalten diese Quellen. `website/_examples.yml` und `scripts/website_assets.py` definieren die verwendeten Daten und Downloads.

In einer frischen Arbeitskopie die bestehenden Abhängigkeiten mit `python -m pip install -e ".[publication]"` bereitstellen; außerdem Quarto und Chrome/Chromium für Kaleido. Anschließend `quarto render website` ausführen. Die Pre-render-Schritte verwenden `website_assets.py` und `publication_example.py`; dadurch bleiben auch die bereits veröffentlichten synthetischen Grafik-URLs aus einem frischen Quellenstand erreichbar. Der Sprachwechsel verweist auf `en/index.html`, da Englisch ein eigenes Quarto-Projekt ist; so wird dessen QMD-Quelle nicht als Ressource kopiert. Die folgende Zelle führt genau diesen Build aus und prüft Seitenumfang, Sprachwechsel und die Trennung von Quellen und Ausgabe. Sie veröffentlicht nichts.


In [ ]:
import subprocess
subprocess.run(["quarto", "render", "website"], cwd=ROOT, check=True)
site = ROOT / "website/_site"
for language in ("", "en"):
    sources = list((ROOT / "website" / language).glob("*.qmd"))
    assert sources
    for source in sources:
        assert (site / language / source.with_suffix(".html").name).is_file(), source
assert not list(site.rglob("*.qmd")), "Quarto-Quellen gehören nicht in die Veröffentlichung"
assert 'href="./en/index.html"' in (site / "index.html").read_text(encoding="utf-8") or 'href="en/index.html"' in (site / "index.html").read_text(encoding="utf-8")
assert (site / "assessment.html").is_file()
print("Website einschließlich Bewertungstutorial erstellt; Veröffentlichung bleibt separat.")

assert (site / "assessment-explained.html").is_file()
assert (site / "assets/assessment-explained/index.html").is_file()
for name in ("howto_assess", "howto_assessment_explained"):
    assert (site / "downloads" / f"{name}.ipynb").is_file()


## Auf GitHub Pages veröffentlichen

Zuerst eine laufende Quarto-Vorschau beenden und `quarto render website` ausführen. Der Build erzeugt Deutsch und Englisch gemeinsam in `website/_site`. Danach `quarto publish gh-pages website --no-render --no-prompt --no-browser` ausführen. Quarto veröffentlicht die fertigen Dateien auf dem Branch `gh-pages`. GitHub Pages verwendet diesen Branch und den Ordner `/`.

Öffentliche URLs: https://simonschaluppe.github.io/peexcel/ und https://simonschaluppe.github.io/peexcel/en/. Voraussetzung ist ein GitHub-Konto mit Schreibzugriff auf das Repository. Die folgende Zelle veröffentlicht nur, wenn `PUBLISH` ausdrücklich aktiviert wird.


In [ ]:
import subprocess
PUBLISH = False
command = ["quarto", "publish", "gh-pages", "website", "--no-render", "--no-prompt", "--no-browser"]
if PUBLISH:
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Nach erfolgreichem Rendern: " + " ".join(command))


## Interaktive Einbettungen vorübergehend deaktiviert

Die deutschen und englischen Seiten `examples`, `envelope` und `components` laden derzeit keine interaktiven Grafik-Iframes. Die ursprünglichen Einbettungen sind im QMD auskommentiert. Auf der Beispielseite ersetzen die vorhandenen SVG-Exporte die vier Jahresbilanzen und die saisonalen F7-Profile. Der vollständige Stundenverlauf und die Schnittansichten bleiben über separate Links erreichbar und laden erst beim Öffnen. Es werden keine zusätzlichen Grafikdateien oder Datenformate gepflegt.

Für die Prüfung den bestehenden Build `quarto render website` ausführen. Im gerenderten Inhalt dieser sechs Seiten dürfen keine `iframe.pexl-chart` oder `iframe.peq-lesson` vorhanden sein; die SVG-Dateien und Links müssen erreichbar sein. Zur Reaktivierung die betreffenden QMD-Kommentare entfernen und die statischen Ersatzabbildungen entfernen.


## Das Bewertungs-Notebook als Quarto-Seite

`notebooks/howto_assess.ipynb` ist die einzige bearbeitbare Inhaltsquelle des Bewertungstutorials. Texte und Python-Code werden dort geändert. Nach Code- oder Datenänderungen das Notebook ausführen und mit seinen Ausgaben speichern.

Der vorhandene Pre-render-Schritt `scripts/website_assets.py` ruft `prepare_assessment()` auf. Diese Funktion erzeugt `website/assessment.ipynb` mit Seitentitel, Downloadlink und den gespeicherten Ausgaben. Die Kopie ist ignoriert und wird bei jedem Build überschrieben; sie wird nicht manuell gepflegt. Der Zweck dieser einen abgeleiteten Datei ist, das außerhalb des Website-Projekts liegende Notebook mit dessen Navigation und Gestaltung nativ zu rendern. Es gibt keine zweite Text- oder Berechnungsquelle und kein neues Datenschema. Der Download wird wie die übrigen Downloads ohne gespeicherte Ausgaben erzeugt.

In `website/_quarto.yml` steht unter `project.render` zusätzlich `assessment.ipynb`. Derselbe Dateiname steht in der Seitenleiste unter „Dokumentation und Referenzen“. Quarto rendert Markdown, aufklappbaren Python-Code, Tabellen und interaktive Plotly-Ausgaben gemeinsam nach `website/_site/assessment.html`. `execute.enabled: false` in der erzeugten Kopie verhindert eine erneute Ausführung beim Rendern. Die öffentliche Seite benötigt keinen Python-Server.

Die folgende Zelle zeigt genau diesen Vorbereitungsschritt. Danach baut `quarto render website` die Website wie gewohnt. Auf „Python-Schnittstelle und Notebooks“ führt ein Link zum Tutorial; die Seite selbst bietet den Notebook-Download. Es wird nichts veröffentlicht.


In [ ]:
from website_assets import prepare_assessment
assessment_page = prepare_assessment(output)
assessment = __import__("nbformat").read(assessment_page, as_version=4)
print("Quarto-Eingabe:", assessment_page)
print("Zellen einschließlich Seitenkopf:", len(assessment.cells))
print("Download:", output / "downloads/howto_assess.ipynb")

## Vom Projektexport zu beiden Bewertungsseiten

Die Website bietet zwei ergänzende Ansichten:

- **Technisches Bewertungs-Tutorial:** `howto_assess.ipynb` wird als `assessment.html` mit Texten, Code und gespeicherten Diagrammen dargestellt.
- **Projekt-Varianten-Explorer:** `howto_assessment_explained.ipynb` liefert die fachlichen Texte; `assessment-explained.html` bettet den interaktiven Explorer ohne technischen Begleittext ein. Die vollständige Ansicht liegt unter `assets/assessment-explained/index.html`.

### 1. Projektexport auswählen

Eine vorhandene PEExcel-Exportdatei mit gespeicherten Szenarien verwenden. Der Explorer berechnet keine neuen Varianten. Die Codezelle unten erzeugt eine eigenständige HTML-Datei mit allen auswählbaren Szenarien. Bei eigenen Projekten `QUELLE` ersetzen. Die Vorauswahl nutzt passende Lehrbeispiele, sonst die ersten drei Szenarien. Der tatsächliche Dateiname erscheint als Datengrundlage.

In der Kommandozeile ist derselbe Ablauf möglich:

```powershell
python scripts/assessment_explained.py --source "C:/Projekte/Projektexport.xlsx" --output "notebooks/local_exports/mein-projekt/index.html"
```

Die neue Option `source` verwendet denselben Import und dieselben Plotfunktionen. Es wird keine zusätzliche Projektkonfiguration benötigt. Fachliche Texte werden im erklärenden Notebook bearbeitet; ein eigener Projektexport ersetzt diese Texte nicht automatisch.


In [2]:
from assessment_explained import build as build_explorer

QUELLE = ROOT / "data/exports/Klimsta_Abschlussprojekt_Varianten_V3.xlsx"
explorer_html = build_explorer(
    ROOT / "notebooks/local_exports/projekt_varianten_explorer/index.html",
    source=QUELLE,
)
assert explorer_html.is_file()
print(explorer_html)

C:\Users\schneids\code\peexcel\notebooks\local_exports\projekt_varianten_explorer\index.html


### 2. Technisches Notebook mit demselben Export ausführen

In `notebooks/howto_assess.ipynb` die Variable `QUELLE` auf dieselbe Exportdatei setzen. Das Notebook zeigt zunächst die vorhandenen Varianten. Anschließend `AUSWAHL` an deren tatsächliche Namen anpassen, alle Zellen ausführen und mit Ausgaben speichern. Diese gespeicherten Ausgaben übernimmt Quarto; der Website-Build führt das technische Notebook nicht nochmals aus.

### 3. Beide Ansichten in PEExcel-Pages bauen

```powershell
quarto render website
```

Der vorhandene Pre-render-Schritt `website_assets.py` erzeugt nun auch den Explorer sowie beide Notebook-Downloads. Die Navigation enthält den Explorer unter „Analysieren und Bewerten“; unter „Python-Schnittstelle und Notebooks“ sind beide Ansichten verlinkt.

Der reguläre Website-Build verwendet den mitgelieferten Lehrprojektexport. Soll die Website dauerhaft einen anderen Export zeigen, in `scripts/website_assets.py` beim Aufruf von `build_explorer(...)` zusätzlich `source=ROOT / "data/exports/MeinProjekt.xlsx"` setzen. Die Quelldatei muss beim Build verfügbar sein. Ein zuvor manuell erzeugtes Explorer-HTML wird vom regulären Build ersetzt.

Zu prüfen sind `website/_site/assessment.html`, `website/_site/assessment-explained.html` und `website/_site/assets/assessment-explained/index.html`. Beide Downloads liegen unter `website/_site/downloads/`.

### 4. Nach Prüfung veröffentlichen

```powershell
quarto publish gh-pages website --no-render
```

Dieser Befehl veröffentlicht die fertig gebaute Website. Der Build und die Beispielzelle oben veröffentlichen nichts. Die eigenständige Explorer-HTML-Datei kann ebenso in einen anderen Quarto-Bericht kopiert werden; der konkrete Weg für `peq-higrube` steht in `docs/development/assessment_explained.md`.
